# OdiaBench — Model Evaluation Notebook

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tripathysagar/odia_eval/blob/main/notebooks/eval_odiabench.ipynb)

Evaluates any 4-bit quantised model against the 5 translated OdiaBench reasoning benchmarks.

**Supported hardware (auto-detected):**
- Kaggle 2×T4 (32 GB total)
- Colab Pro / Colab Pro+ (A100 40 GB or 80 GB)
- Kaggle single T4 or L4 (16–24 GB)

**Default model:** `google/gemma-4-E2B-it` (Gemma 4 2B IT, 4-bit via BitsAndBytes)  
Swap `MODEL_ID` in the config cell to evaluate any model.

> Needs a GPU to finish in reasonable time. For a **pure-CPU, no-model** walkthrough of the `odia_eval` library API, open [`capability.ipynb`](./capability.ipynb) instead.

## 1 — Install dependencies

In [ ]:
%%capture
# Install odia_eval (latest) + model deps (transformers, bnb, accelerate) + datasets
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-qqq", "--no-cache-dir",
     "odia_eval[model] @ git+https://github.com/tripathysagar/odia_eval.git",
     "datasets", "tqdm"],
    check=False,
)

## 2 — Clone OdiaBench (for benchmark data) and verify `odia_eval`

In [ ]:
import os
from pathlib import Path

# Detect runtime root (Kaggle vs Colab vs local)
if Path("/kaggle/working").exists():
    WORKDIR = Path("/kaggle/working")
elif Path("/content").exists():
    WORKDIR = Path("/content")
else:
    WORKDIR = Path(".").resolve()

# Clone OdiaBench for the benchmark data files (dataset/data/*.jsonl)
REPO_DIR = WORKDIR / "OdiaBench"
if not REPO_DIR.exists():
    os.system(f"git clone --depth=1 https://github.com/tripathysagar/OdiaBench {REPO_DIR}")
else:
    print(f"Repo already at {REPO_DIR}, skipping clone")

DATA_DIR = str(REPO_DIR / "dataset" / "data")

# odia_eval was pip-installed in the previous cell
import odia_eval
from odia_eval import BENCHMARKS, run_eval, run_all, full_report, to_records
print(f"odia_eval {odia_eval.__version__} installed. Benchmarks: {BENCHMARKS}")
print(f"Data dir: {DATA_DIR}")

## 3 — Hardware detection + batch-size config

In [ ]:
import torch

def detect_hardware() -> dict:
    """Return GPU info and a single uniform batch size for a ~2B 4-bit model.

    One batch size is used across every benchmark (sized for the GSM8K
    long-generation case so MCQs are merely under-utilised, never OOM).
    Override the returned value by editing ``BATCH_SIZE`` below.
    """
    if not torch.cuda.is_available():
        print("WARNING: no GPU detected — eval will be very slow on CPU")
        return dict(n_gpus=0, gpu_name="CPU", vram_gb=0, batch_size=2)

    n_gpus      = torch.cuda.device_count()
    gpu_name    = torch.cuda.get_device_name(0)
    vram_per_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    total_vram  = n_gpus * vram_per_gb

    # Conservative single-knob heuristic for Gemma 4 2B IT NF4
    # (model weights ~1.5 GB, GSM8K max_new_tokens=512 dominates KV cache).
    if total_vram >= 70:         # A100 80 GB / H100
        batch_size = 256
    elif total_vram >= 35:       # A100 40 GB  OR  2×T4 (32 GB)
        batch_size = 128
    elif total_vram >= 20:       # L4 (24 GB) / A10G (24 GB)
        batch_size = 64
    elif total_vram >= 14:       # single T4 (16 GB)
        batch_size = 64
    else:                        # <16 GB fallback
        batch_size = 8

    return dict(
        n_gpus=n_gpus,
        gpu_name=gpu_name,
        vram_per_gb=round(vram_per_gb, 1),
        total_vram_gb=round(total_vram, 1),
        batch_size=batch_size,
    )

HW = detect_hardware()
# Single uniform batch size, applied to every benchmark.
# Common values: 64 (T4/L4), 128 (A100-40G / 2×T4), 256 (A100-80G/H100).
BATCH_SIZE = HW["batch_size"]

print(f"GPU : {HW['gpu_name']}  ×{HW['n_gpus']}")
print(f"VRAM: {HW.get('total_vram_gb', 0):.1f} GB  "
      f"(per-GPU: {HW.get('vram_per_gb', 0):.1f} GB)")
print(f"Batch size (uniform across all benchmarks): {BATCH_SIZE}")

## 4 — Config  *(edit here)*

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────
MODEL_ID   = "google/gemma-4-E2B-it"    # Gemma 4 2B IT (official HF repo)
# MODEL_ID = "Qwen/Qwen3-0.6B-Instruct" # swap to evaluate the student model
# MODEL_ID = "sarvam-ai/sarvam-1"        # Indic-specialist baseline

LOAD_IN_4BIT = True   # NF4 quantisation via BitsAndBytes

# ── torch.compile (inference speedup) ─────────────────────────────────────
# Compiles model.forward via TorchDynamo + Inductor → CUDA graphs.  On
# Ampere+ GPUs (A100/L4/H100) this is typically a 1.3–2x throughput win
# on long-generation tasks; on Turing (T4) the gain is smaller.  First
# forward pass after compile is slow (graph capture); subsequent calls
# run on the cached compiled graph.
USE_TORCH_COMPILE   = True
TORCH_COMPILE_MODE  = "reduce-overhead"  # "default" | "reduce-overhead" | "max-autotune"
# ── Attention backend ─────────────────────────────────────────────────────
# "auto" tries flash_attention_2 → sdpa → default (silent fallback).
# Pass an explicit string to force one backend.
ATTN_IMPL = "auto"

# ── Static KV-cache + fullgraph compile (opt-in) ─────────────────────────
# Requires USE_TORCH_COMPILE=True and transformers≥4.40.  Often fails with
# BnB 4-bit + variable batch shapes — the loader skips with a warning.
USE_STATIC_CACHE = False

# ── Length-bucketed batching ──────────────────────────────────────────────
SORT_BY_LENGTH = False

# ── Reasoning prompts ─────────────────────────────────────────────────────
# If True, wrap prompts with <think>+\boxed{...} instructions.
# Also disables the \n\n early-stop heuristic in MCQ generation (paragraphs
# in the thinking block must not truncate output).
REASONING = False

# ── Self-consistency (test-time compute) ─────────────────────────────────
# N_VOTES > 1 samples N completions per prompt and majority-votes the
# extracted answer.  Requires generate_fn to be STOCHASTIC (do_sample=True)
# — greedy decoding produces identical votes.  Actual batch sent to
# generate_fn becomes batch_size * N_VOTES, so halve BATCH_SIZE if you
# bump N_VOTES on a memory-tight GPU.
N_VOTES = 1

# Per-benchmark early-stop flags (recorded in .meta.json; gsm8k never uses box stop).
_EARLY_STOP_ON_BOX = {
    "gsm8k":      False,
    "arc":        True,
    "truthfulqa": True,
    "winogrande": True,
    "hellaswag":  True,
}


# ── Eval ──────────────────────────────────────────────────────────────────
N_SAMPLES  = 200      # rows per benchmark (None = full split)
SEED       = 42       # random seed — same seed → same rows across models

# Benchmarks to run: remove "hellaswag" for a quick 4-benchmark smoke test
RUN_BENCHMARKS = ["arc", "gsm8k", "truthfulqa", "winogrande", "hellaswag"]

# Output file
RESULTS_JSONL = WORKDIR / f"odiabench_results_{MODEL_ID.replace('/', '_')}.jsonl"

print(f"Model        : {MODEL_ID}")
print(f"Benchmarks   : {RUN_BENCHMARKS}")
print(f"Samples/bench: {N_SAMPLES}")
print(f"Seed         : {SEED}")
print(f"torch.compile: {USE_TORCH_COMPILE} (mode={TORCH_COMPILE_MODE!r})")
print(f"Results to   : {RESULTS_JSONL}")
print(f"attn impl    : {ATTN_IMPL!r}")
print(f"static cache : {USE_STATIC_CACHE}")
print(f"sort_by_len  : {SORT_BY_LENGTH}")
print(f"reasoning    : {REASONING}")
print(f"n_votes      : {N_VOTES}")

## 5 — Load model (4-bit NF4, multi-GPU aware)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit             = LOAD_IN_4BIT,
    bnb_4bit_quant_type      = "nf4",
    bnb_4bit_compute_dtype   = torch.float16,
    bnb_4bit_use_double_quant= True,
) if LOAD_IN_4BIT else None

# Try AutoProcessor first (Gemma 4 needs it), fall back to AutoTokenizer
try:
    tokenizer = AutoProcessor.from_pretrained(MODEL_ID)
    print(f"Loaded processor: {type(tokenizer).__name__}")
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    print(f"Loaded tokenizer: {type(tokenizer).__name__}")


_model_kwargs = dict(
    quantization_config = bnb_cfg,
    device_map          = "auto",
    torch_dtype         = torch.float16,
)

_attn_chain = (
    ["flash_attention_2", "sdpa", None]
    if ATTN_IMPL == "auto"
    else [ATTN_IMPL]
)
_attn_used = "default"
model = None
for _impl in _attn_chain:
    try:
        if _impl is None:
            model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
            _attn_used = "default"
        else:
            model = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, attn_implementation=_impl, **_model_kwargs,
            )
            _attn_used = _impl
        break
    except (ValueError, TypeError, ImportError, OSError):
        if ATTN_IMPL != "auto":
            raise
        continue
if model is None:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
    _attn_used = "default"
model.eval()
print(f"Loaded: {MODEL_ID}  |  attn: {_attn_used}")


# Unwrap inner tokenizer if we got a Processor (needed for pad/eos/decode)
_inner_tok = getattr(tokenizer, "tokenizer", tokenizer)

if _inner_tok.pad_token is None:
    _inner_tok.pad_token = _inner_tok.eos_token
_inner_tok.padding_side = "left"

DEVICE = next(model.parameters()).device
print(f"Model device : {DEVICE}")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM used    : {allocated:.2f} GB")


# ── torch.compile model.forward ──────────────────────────────────────────
# Compile only model.forward (model.generate has Python control flow that
# trips up Dynamo).  USE_STATIC_CACHE switches to fullgraph + static KV.
_torch_compile_applied = False
_static_cache_applied = False
if USE_TORCH_COMPILE:
    _torch_ver = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    if _torch_ver < (2, 4):
        print(f"[warn] torch.compile skipped: torch {torch.__version__} < 2.4")
    else:
        _compile_kwargs = dict(
            mode=TORCH_COMPILE_MODE,
            dynamic=True,
            fullgraph=False,
        )
        if USE_STATIC_CACHE:
            try:
                model.generation_config.cache_implementation = "static"
                _compile_kwargs = dict(
                    mode=TORCH_COMPILE_MODE,
                    dynamic=False,
                    fullgraph=True,
                )
                _static_cache_applied = True
                print("static KV-cache: generation_config.cache_implementation='static'")
            except Exception as e:
                print(f"[warn] static KV-cache skipped ({type(e).__name__}: {e}); "
                      "BnB 4-bit + variable batch shapes often break fullgraph compile")
        try:
            model.forward = torch.compile(model.forward, **_compile_kwargs)
            _torch_compile_applied = True
            print(f"torch.compile: applied to model.forward  "
                  f"(mode={TORCH_COMPILE_MODE!r}, "
                  f"dynamic={_compile_kwargs['dynamic']}, "
                  f"fullgraph={_compile_kwargs['fullgraph']})")
            print("  First forward pass will be slow (graph capture); "
                  "subsequent calls run on the compiled graph.")
        except Exception as e:
            if _static_cache_applied:
                print(f"[warn] fullgraph compile failed ({type(e).__name__}: {e}); "
                      "retrying without static cache")
                _static_cache_applied = False
                try:
                    model.generation_config.cache_implementation = None
                except Exception:
                    pass
                try:
                    model.forward = torch.compile(
                        model.forward,
                        mode=TORCH_COMPILE_MODE,
                        dynamic=True,
                        fullgraph=False,
                    )
                    _torch_compile_applied = True
                    print("torch.compile: applied (fallback dynamic=True, fullgraph=False)")
                except Exception as e2:
                    print(f"[warn] torch.compile failed ({type(e2).__name__}: {e2}); "
                          "falling back to eager mode")
            else:
                print(f"[warn] torch.compile failed ({type(e).__name__}: {e}); "
                      "falling back to eager mode")
else:
    print("torch.compile: disabled (USE_TORCH_COMPILE=False)")


## 6 — Define `generate_fn`

In [ ]:
import re

from transformers import ProcessorMixin, StoppingCriteria, StoppingCriteriaList

_is_processor = isinstance(tokenizer, ProcessorMixin)


def _apply_chat_template(prompts: list[str]) -> list[str]:
    """Wrap each prompt in the model's chat template."""
    tpl = _inner_tok if hasattr(_inner_tok, "apply_chat_template") else tokenizer
    if not hasattr(tpl, "apply_chat_template"):
        return prompts
    try:
        return [
            tpl.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False,
                add_generation_prompt=True,
            )
            for p in prompts
        ]
    except Exception:
        return prompts


def _tokenize(texts: list[str]):
    """Tokenize handling both Processor (Gemma 4) and plain Tokenizer.

    For Processors we filter to input_ids + attention_mask only, because
    the Processor may also return pixel_values=None or other keys that
    would confuse model.generate(**enc).
    """
    if _is_processor:
        raw = tokenizer(text=texts, return_tensors="pt",
                        padding=True, truncation=True, max_length=1024)
        from transformers import BatchEncoding
        return BatchEncoding({
            k: v for k, v in raw.items()
            if k in ("input_ids", "attention_mask") and v is not None
        })
    return _inner_tok(texts, return_tensors="pt",
                      padding=True, truncation=True, max_length=1024)


_BOXED_CLOSED_RE = re.compile(r"\\boxed\s*\{\s*[^{}]*?\}")


class _McqEarlyStopCriteria(StoppingCriteria):
    """Halt when every row closes \\boxed{...} (always) or emits a blank line (plain MCQ only)."""

    _DECODE_TAIL = 64

    def __init__(
        self,
        tok,
        prompt_len: int,
        check_every: int = 4,
        *,
        reasoning: bool = False,
    ) -> None:
        self.tok = tok
        self.prompt_len = prompt_len
        self.check_every = check_every
        self.reasoning = reasoning
        self._step = 0
        self._done: list[bool] | None = None

    def _row_done(self, text: str) -> bool:
        if not self.reasoning and "\n\n" in text:
            return True
        return bool(_BOXED_CLOSED_RE.search(text))

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        self._step += 1
        if self._step % self.check_every != 0:
            return False
        n = input_ids.shape[0]
        if self._done is None:
            self._done = [False] * n
        new_ids = input_ids[:, self.prompt_len:]
        for i in range(n):
            if self._done[i]:
                continue
            tail = new_ids[i, -self._DECODE_TAIL:].tolist()
            if self._row_done(self.tok.decode(tail, skip_special_tokens=True)):
                self._done[i] = True
        return all(self._done)


def make_generate_fn(
    max_new_tokens: int = 512,
    early_stop_on_box: bool = False,
    reasoning: bool = False,
):
    @torch.inference_mode()
    def _generate(prompts: list[str]) -> list[str]:
        templated = _apply_chat_template(prompts)
        enc = _tokenize(templated).to(DEVICE)

        gen_kwargs = dict(
            max_new_tokens  = max_new_tokens,
            do_sample       = False,
            temperature     = 1.0,
            pad_token_id    = _inner_tok.pad_token_id or _inner_tok.eos_token_id,
            eos_token_id    = _inner_tok.eos_token_id,
        )
        if early_stop_on_box:
            prompt_len = enc["input_ids"].shape[1]
            gen_kwargs["stopping_criteria"] = StoppingCriteriaList([
                _McqEarlyStopCriteria(
                    _inner_tok, prompt_len, reasoning=reasoning,
                ),
            ])

        out = model.generate(**enc, **gen_kwargs)

        prompt_len = enc["input_ids"].shape[1]
        gen_ids    = out[:, prompt_len:]
        return _inner_tok.batch_decode(gen_ids, skip_special_tokens=True)

    return _generate


generate_mcq   = make_generate_fn(max_new_tokens=32, early_stop_on_box=True, reasoning=REASONING)
generate_gsm8k = make_generate_fn(max_new_tokens=512, early_stop_on_box=False, reasoning=REASONING)

print("generate_fn ready")

test_out = generate_mcq(["ଏହି ବାକ୍ୟ ର ଅର୍ଥ କ'ଣ?"])
print("Sanity check output:", repr(test_out[0][:80]))


## 7 — Run evaluation

In [ ]:
# Per-benchmark generate function + batch size.
# GSM8K generates up to 512 tokens (chain-of-thought) so it needs a
# smaller batch to avoid OOM; MCQ benchmarks generate ≤32 tokens.
_BENCH_CFG = {
    "gsm8k":      (generate_gsm8k, max(1, BATCH_SIZE // 4)),
    "arc":        (generate_mcq,   BATCH_SIZE),
    "truthfulqa": (generate_mcq,   BATCH_SIZE),
    "winogrande": (generate_mcq,   BATCH_SIZE),
    "hellaswag":  (generate_mcq,   BATCH_SIZE),
}

all_results = {}

for benchmark in RUN_BENCHMARKS:
    gen_fn, bs = _BENCH_CFG[benchmark]
    print(f"\n{'='*50}")
    print(f" {benchmark.upper()}  |  n_samples={N_SAMPLES}  |  batch_size={bs}")
    print(f"{'='*50}")

    all_results[benchmark] = run_eval(
        benchmark,
        gen_fn,
        n_samples  = N_SAMPLES,
        seed       = SEED,
        batch_size = bs,
        data_dir   = DATA_DIR,
        verbose        = True,
        sort_by_length = SORT_BY_LENGTH,
        reasoning      = REASONING,
        n_votes        = N_VOTES,
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 8 — Results

In [ ]:
print(f"\nModel: {MODEL_ID}\n")
print(full_report(all_results))

## 9 — Save results

In [ ]:
import json
from datetime import datetime, timezone
from odia_eval import save_results, load_results

# Per-row results (prompt + gold + extracted + prediction + correct)
n_rows = save_results(all_results, RESULTS_JSONL)
print(f"Saved {n_rows} row-level records → {RESULTS_JSONL}")

# Companion metadata sidecar (one row, easy to grep across runs)
meta = {
    "model_id":   MODEL_ID,
    "n_samples":  N_SAMPLES,
    "seed":       SEED,
    "batch_size": BATCH_SIZE,
    "load_4bit":  LOAD_IN_4BIT,
    "torch_compile":         bool(_torch_compile_applied),
    "torch_compile_mode":    TORCH_COMPILE_MODE if _torch_compile_applied else None,
    "attn_implementation":   _attn_used,
    "use_static_cache":      bool(_static_cache_applied),
    "reasoning":             REASONING,
    "n_votes":               N_VOTES,
    "early_stop_on_box":     dict(_EARLY_STOP_ON_BOX),
    "sort_by_length":        SORT_BY_LENGTH,
    "odia_eval_version":     odia_eval.__version__,
    "gpu":        HW["gpu_name"],
    "n_gpus":     HW["n_gpus"],
    "timestamp":  datetime.now(timezone.utc).isoformat(),
}
meta_path = str(RESULTS_JSONL).rsplit(".", 1)[0] + ".meta.json"
with open(meta_path, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)
print(f"Saved run metadata    → {meta_path}")

# Sanity: round-trip the results and confirm the report is identical
reloaded = load_results(RESULTS_JSONL)
assert full_report(all_results) == full_report(reloaded), "round-trip mismatch"
print("Round-trip OK — re-score later with load_results() without re-running generation.")

## 10 — Per-benchmark breakdown (optional deeper look)

In [ ]:
# Show a few wrong predictions to spot failure modes
for bm, rows in all_results.items():
    wrong = [r for r in rows if not r.correct][:3]
    if not wrong:
        continue
    print(f"\n── {bm} — first 3 wrong ──")
    for r in wrong:
        print(f"  id={r.row_id}  extracted={r.score_result.extracted!r}  "
              f"gold={r.score_result.gold!r}")
        print(f"  prediction: {r.score_result.prediction[:120]!r}")